# 2021 Municipal Election Results: Data Cleaning

**Goal:** turn the raw IEC ward-level results for 2021 into one clean table with one row per municipality.

**Input:** the provincial CSV files in `MyDrive/DIRISA_SDC/data/raw/iec_lge/2021/`
**Output:** `MyDrive/DIRISA_SDC/data/interim/results_municipal_2021_clean.csv`

| Step | What happens |
|------|--------------|
| 1 | Load every provincial CSV and combine them into one table |
| 2 | Keep Ward ballots only |
| 3 | Reduce to one row per voting station |
| 4 | Aggregate to municipality level |
| 5 | Calculate turnout |
| 6 | Sanity checks |
| 7 | Save the clean file |

The same steps are used for 2011, 2016 and 2021. Only `YEAR` in the setup cell changes.

## Setup

In [ ]:
import re
from pathlib import Path

import pandas as pd
from google.colab import drive

# Connect Google Drive so the notebook can read the raw files
drive.mount('/content/drive')

# ---- Settings (the only thing that differs between the notebooks) ----
YEAR = 2021

# ---- Paths ----
DRIVE   = Path("/content/drive/MyDrive")
RAW     = DRIVE / "DIRISA_SDC" / "data" / "raw" / "iec_lge" / str(YEAR)   # raw provincial CSV files
INTERIM = DRIVE / "DIRISA_SDC" / "data" / "interim"   # where clean files are saved
INTERIM.mkdir(parents=True, exist_ok=True)

print(f"Cleaning year: {YEAR}")
print("Raw folder exists:", RAW.exists())

## Step 1: Load and combine the provincial files

The IEC publishes one CSV per province. Some files are saved in different text encodings, so the reader below tries several encodings until the header reads properly.

In [ ]:
def read_csv_any_encoding(path):
    """Read a CSV, trying common encodings until the header looks correct."""
    for enc in ("utf-8", "utf-8-sig", "utf-16", "latin-1"):
        try:
            df = pd.read_csv(path, encoding=enc)
            # A correctly read file has several columns, and its first column
            # name does not start with "ÿ" (a sign the encoding was wrong).
            if df.shape[1] > 1 and not str(df.columns[0]).startswith("ÿ"):
                return df
        except (UnicodeDecodeError, UnicodeError):
            continue
    # Last resort: accept whatever latin-1 gives us
    return pd.read_csv(path, encoding="latin-1")


province_files = sorted(RAW.glob("*.csv"))
print(f"{len(province_files)} province file(s) found:")
for f in province_files:
    print(" -", f.name)

# Files such as "KN (1).csv" are accidental re-downloads. Left in the folder,
# they would be counted twice and inflate every total.
duplicates = [f.name for f in province_files if re.search(r" \(\d+\)\.csv$", f.name)]
if duplicates:
    print("\nWARNING: possible duplicate downloads, remove them from the folder:", duplicates)

# Stack all provinces into one table, remembering which file each row came from
raw = pd.concat(
    [read_csv_any_encoding(f).assign(source_file=f.name) for f in province_files],
    ignore_index=True,
    sort=False,
)

print(f"\nCombined shape: {raw.shape}")
print("Columns:", list(raw.columns))
raw.head()

## Step 2: Keep Ward ballots only

Each voting station reports more than one ballot type (e.g. Ward and Proportional Representation). Mixing them would count the same voters more than once, so only the **Ward** ballot is kept.

In [ ]:
print("Ballot types in the raw data:")
print(raw["BallotType"].value_counts())

ward = raw[raw["BallotType"] == "Ward"].copy()
print(f"\nRows after keeping Ward only: {len(ward)} (was {len(raw)})")

## Step 3: One row per voting station

The file has one row per **party per station**, so `RegisteredVoters` and `SpoiltVotes` are repeated on every party row of the same station. Summing them directly would multiply them by the number of parties. Keeping one row per station fixes this.

(`TotalValidVotes` is different: it is a per-party count, so it is summed from the full Ward table in Step 4.)

In [ ]:
one_row_per_station = ward.drop_duplicates(
    subset=["Province", "Municipality", "VotingStationName"]
)
print(f"Voting stations (Ward ballot): {len(one_row_per_station)}")

## Step 4: Aggregate to municipality level

In [ ]:
# Registered voters and spoilt votes: sum over stations (one row each)
registered_and_spoilt = (
    one_row_per_station
    .groupby(["Province", "Municipality"])[["RegisteredVoters", "SpoiltVotes"]]
    .sum()
    .reset_index()
)

# Valid votes: sum over every party row of the Ward ballot
valid_votes = (
    ward
    .groupby(["Province", "Municipality"])["TotalValidVotes"]
    .sum()
    .reset_index()
)

# Join the two tables (a municipality must appear in both to be kept)
muni = registered_and_spoilt.merge(valid_votes, on=["Province", "Municipality"], how="inner")
print(f"Municipalities: {len(muni)}")

## Step 5: Calculate turnout

Turnout = (valid votes + spoilt votes) / registered voters × 100

In [ ]:
muni["total_votes_cast"] = muni["TotalValidVotes"] + muni["SpoiltVotes"]
muni["turnout_pct"] = muni["total_votes_cast"] / muni["RegisteredVoters"] * 100
muni["year"] = YEAR

# Fixed column order so the 2011, 2016 and 2021 files line up
muni = muni[[
    "year", "Province", "Municipality",
    "RegisteredVoters", "TotalValidVotes", "SpoiltVotes",
    "total_votes_cast", "turnout_pct",
]]

muni.head(10)

## Step 6: Sanity checks

Turnout should sit between 0% and 100%, and the national figure should look realistic for a South African local election.

In [ ]:
# 1. Spread of municipal turnout
print(muni["turnout_pct"].describe())

# 2. National turnout (uses the municipal totals, so it ties back to the table)
national_turnout = muni["total_votes_cast"].sum() / muni["RegisteredVoters"].sum() * 100
print(f"\nNational turnout check: {national_turnout:.1f}%")

# 3. Structural checks
print("Rows with turnout outside 0-100%:", ((muni["turnout_pct"] < 0) | (muni["turnout_pct"] > 100)).sum())
print("Missing values:", int(muni.isna().sum().sum()))
assert not muni.duplicated(["Province", "Municipality"]).any(), "Duplicate municipalities found"

## Step 7: Save the clean file

In [ ]:
out_path = INTERIM / f"results_municipal_{YEAR}_clean.csv"
muni.to_csv(out_path, index=False)
print(f"Saved to: {out_path}")